# 空き家が多いのはどこ？ 都道府県データで調べる
## Where are vacant homes most common? A look at Japan's prefectures

**氏名 / Name:** （栗田大雅）

**使用データ / Data:** 授業で配布された `japan.csv`（47都道府県、92列）。年や出典の詳細は未確認のため、「現在の日本」とは断定せず、このCSVに入っている値として説明します。

## 自分の問い / My question

**問い：** 空き家の「件数」ではなく「割合」で見ると、どの都道府県で空き家が多いのか。高齢化との関係はあるか。

**予想：** 人口が多い都市部ほど空き家も多いと思う。

*Question: Measured as a share of all homes (not a raw count), which prefectures have the most vacant homes, and is it related to ageing?*

## 1. 準備と読み込み / Setup and load

In [1]:
import pandas as pd

df = pd.read_csv('japan.csv')
print(df.shape)

(47, 92)


## 2. データを確認する / Inspect the data

使う列だけに絞って `head()`・`info()`・`describe()` で確認します。`空き家数`と`総住宅数`は戸数、`65歳以上人口`と`総人口`は人です。

In [2]:
columns = ['都道府県', '総人口', '65歳以上人口', '総住宅数', '空き家数']
data = df[columns]
data.head()

,都道府県,総人口,65歳以上人口,総住宅数,空き家数
0,北海道,5140000,1686000,2807200,379800
1,青森県,1204000,419000,592400,88700
2,岩手県,1181000,408000,579300,93500
3,宮城県,2280000,659000,1089300,130500
4,秋田県,930000,359000,445700,60800


In [3]:
data.info(memory_usage=False)

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   都道府県     47 non-null     str  
 1   総人口      47 non-null     int64
 2   65歳以上人口  47 non-null     int64
 3   総住宅数     47 non-null     int64
 4   空き家数     47 non-null     int64
dtypes: int64(4), str(1)

In [4]:
data.describe().round(2)

,総人口,65歳以上人口,総住宅数,空き家数
count,47.00,47.00,47.00,47.00
mean,2658425.53,770957.45,1327821.28,180614.89
std,2793536.00,693147.11,1451567.65,165207.62
min,544000.00,180000.00,256600.00,39900.00
25%,1046500.00,348000.00,511750.00,82700.00
50%,1563000.00,523000.00,813700.00,126800.00
75%,2655000.00,790500.00,1333600.00,197250.00
max,14038000.00,3202000.00,7671600.00,809900.00


## 3. forループで割合を計算する / Compute shares with a loop

各都道府県について、空き家率 = 空き家数 ÷ 総住宅数 × 100、高齢化率 = 65歳以上人口 ÷ 総人口 × 100 を計算し、リストに集めて表にします。

In [5]:
results = []

for index, row in df.iterrows():
    vacancy_rate = row['空き家数'] / row['総住宅数'] * 100
    old_share = row['65歳以上人口'] / row['総人口'] * 100
    results.append({
        '都道府県': row['都道府県'],
        '空き家数': row['空き家数'],
        '空き家率(%)': round(vacancy_rate, 2),
        '高齢化率(%)': round(old_share, 2),
    })

result = pd.DataFrame(results)
result.head()

,都道府県,空き家数,空き家率(%),高齢化率(%)
0,北海道,379800,13.53,32.80
1,青森県,88700,14.97,34.80
2,岩手県,93500,16.14,34.55
3,宮城県,130500,11.98,28.90
4,秋田県,60800,13.64,38.60


## 4. 空き家率の高い5件・低い5件 / Highest and lowest five

In [6]:
ranked = result.sort_values('空き家率(%)', ascending=False)

print('--- 空き家率が高い5件 / Highest ---')
for index, row in ranked.head(5).iterrows():
    print(f"{row['都道府県']}: {row['空き家率(%)']:.1f}%  (高齢化率 {row['高齢化率(%)']:.1f}%)")

print()
print('--- 空き家率が低い5件 / Lowest ---')
for index, row in ranked.tail(5).iterrows():
    print(f"{row['都道府県']}: {row['空き家率(%)']:.1f}%  (高齢化率 {row['高齢化率(%)']:.1f}%)")

--- 空き家率が高い5件 / Highest ---
山梨県: 21.3%  (高齢化率 31.4%)
和歌山県: 20.3%  (高齢化率 34.0%)
長野県: 19.6%  (高齢化率 32.5%)
徳島県: 19.5%  (高齢化率 34.9%)
高知県: 19.1%  (高齢化率 36.1%)

--- 空き家率が低い5件 / Lowest ---
愛知県: 11.3%  (高齢化率 25.6%)
神奈川県: 10.8%  (高齢化率 25.8%)
東京都: 10.6%  (高齢化率 22.8%)
沖縄県: 10.4%  (高齢化率 23.4%)
埼玉県: 10.2%  (高齢化率 27.4%)


## 5. ラベルを付けて数える / Label and count

空き家率が17%以上を「高い」、13%未満を「低い」、その間を「ふつう」とラベル付けして、件数を数えます。しきい値は自分で決めたものです。

In [7]:
counts = {'高い': 0, 'ふつう': 0, '低い': 0}
names = {'高い': [], 'ふつう': [], '低い': []}

for index, row in result.iterrows():
    rate = row['空き家率(%)']
    if rate >= 17:
        label = '高い'
    elif rate < 13:
        label = '低い'
    else:
        label = 'ふつう'
    counts[label] += 1
    names[label].append(row['都道府県'])

for label in ['高い', 'ふつう', '低い']:
    print(f"{label}: {counts[label]}件")
print()
print('低い:', '、'.join(names['低い']))

高い: 10件
ふつう: 26件
低い: 11件

低い: 宮城県、山形県、埼玉県、千葉県、東京都、神奈川県、愛知県、滋賀県、京都府、福岡県、沖縄県


## 6. 件数で見ると？ / What about raw counts?

予想（都市部ほど空き家が多い）を確かめるため、空き家の「件数」が多い上位4都府県の空き家率を見ます。

In [8]:
top_count = df.sort_values('空き家数', ascending=False).head(4)

for index, row in top_count.iterrows():
    rate = row['空き家数'] / row['総住宅数'] * 100
    print(f"{row['都道府県']}: 空き家 {row['空き家数']:,}戸、空き家率 {rate:.1f}%")

東京都: 空き家 809,900戸、空き家率 10.6%
大阪府: 空き家 709,400戸、空き家率 15.2%
神奈川県: 空き家 484,700戸、空き家率 10.8%
愛知県: 空き家 393,800戸、空き家率 11.3%


## 7. 発展：高齢化率のグループごとに比べる / Extra: compare by ageing level

高齢化率で3グループ（28%未満・28〜32%・32%以上）に分け、グループごとに空き家数と総住宅数を合計して空き家率を出します。

In [9]:
groups = {
    '28%未満': {'n': 0, 'vacant': 0, 'homes': 0},
    '28〜32%': {'n': 0, 'vacant': 0, 'homes': 0},
    '32%以上': {'n': 0, 'vacant': 0, 'homes': 0},
}

for index, row in df.iterrows():
    old_share = row['65歳以上人口'] / row['総人口'] * 100
    if old_share < 28:
        key = '28%未満'
    elif old_share < 32:
        key = '28〜32%'
    else:
        key = '32%以上'
    groups[key]['n'] += 1
    groups[key]['vacant'] += row['空き家数']
    groups[key]['homes'] += row['総住宅数']

summary = []
for key, g in groups.items():
    summary.append({
        '高齢化率': key,
        '都道府県数': g['n'],
        '空き家率(%)': round(g['vacant'] / g['homes'] * 100, 1),
    })

pd.DataFrame(summary)

,高齢化率,都道府県数,空き家率(%)
0,28%未満,8,11.7
1,28〜32%,16,14.6
2,32%以上,23,15.8


## 8. 確認：元の一行と比べる / Check against one original row

In [10]:
row = df[df['都道府県'] == '山梨県'].iloc[0]
print('元のデータ:', row['空き家数'], '戸 /', row['総住宅数'], '戸')
print(f"手計算: {row['空き家数']} ÷ {row['総住宅数']} × 100 = {row['空き家数'] / row['総住宅数'] * 100:.2f}%")
print('表の値:', result[result['都道府県'] == '山梨県']['空き家率(%)'].iloc[0], '%')

元のデータ: 90000 戸 / 422000 戸
手計算: 90000 ÷ 422000 × 100 = 21.33%
表の値: 21.33 %


## 結果からわかったこと / What I found

- **結果：** 空き家率が最も高いのは山梨県（21.3%）で、和歌山県（20.3%）、長野県（19.6%）が続いた。最も低いのは埼玉県（10.2%）だった。47都道府県を空き家率でラベル付けすると、「高い」10件、「ふつう」26件、「低い」11件になった。
- **予想との比較：** 空き家の件数が最も多いのは東京都（809,900戸）と大阪府（709,400戸）だが、東京都の空き家率は10.6%で全国でも低い方だった。人口が多い都市部ほど空き家が多い、という予想は「件数」ならそのとおりだが、「割合」では当てはまらなかった。割合で高いのは山梨・和歌山・長野・徳島・高知などだった。
- **高齢化との関係：** 高齢化率が32%以上のグループの空き家率は15.8%で、28%未満のグループの11.7%より高かった。高齢化が進んだ県ほど空き家の割合が高い傾向が見られた。
- **確認：** 山梨県の元の行（空き家90,000戸、総住宅数422,000戸）から手計算した21.33%は、表の値と一致した。
- **限界：** この結果は47都道府県の集計値だけで、年や「空き家」の定義（別荘や賃貸の空室を含むかなど）はCSVからは確認できていない。また、高齢化と空き家の関連は見られても、高齢化が空き家の原因だとは言えない。